In [ ]:
# In[1]:

import pandas as pd

# incident window in UTC
window_start = pd.to_datetime("2024-01-25 16:19:12", utc=True)
window_end = pd.to_datetime("2024-01-25 16:49:12", utc=True)

# load metric.csv
metric = pd.read_csv("metric.csv")
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')

# global per-(cmdb_id, kpi_name) summaries: count and global P95 (computed on full series)
metric_summary = (
    metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(count=('value', 'size'), p95_value=('value', lambda s: float(s.quantile(0.95))))
    .sort_values(['cmdb_id','kpi_name'])
)
metric_summary = metric_summary.head(50)

# number of metric rows within incident window (global count)
metric_window_count = metric[(metric['timestamp'] >= window_start) & (metric['timestamp'] <= window_end)].shape[0]

# load trace.csv
trace = pd.read_csv("trace.csv")
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')

trace_summary = (
    trace
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(count=('value', 'size'), p95_value=('value', lambda s: float(s.quantile(0.95))))
    .sort_values(['cmdb_id','trace_name'])
)
trace_summary = trace_summary.head(50)

trace_window_count = trace[(trace['timestamp'] >= window_start) & (trace['timestamp'] <= window_end)].shape[0]

# load log.csv
log = pd.read_csv("log.csv")
log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)
log['value'] = pd.to_numeric(log['value'], errors='coerce')

log_summary = (
    log
    .groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(count=('value', 'size'), p95_value=('value', lambda s: float(s.quantile(0.95))))
    .sort_values(['cmdb_id','log_name'])
)
log_summary = log_summary.head(50)

log_window_count = log[(log['timestamp'] >= window_start) & (log['timestamp'] <= window_end)].shape[0]

# load error_logs.csv
error_logs = pd.read_csv("error_logs.csv")
error_logs['timestamp'] = pd.to_datetime(error_logs['timestamp'], unit='s', utc=True)

error_summary = (
    error_logs
    .groupby('cmdb_id', as_index=False)
    .agg(count=('timestamp', 'size'),
         earliest_timestamp=('timestamp', 'min'),
         latest_timestamp=('timestamp', 'max'))
    .sort_values('cmdb_id')
)
error_summary = error_summary.head(50)

error_window_count = error_logs[(error_logs['timestamp'] >= window_start) & (error_logs['timestamp'] <= window_end)].shape[0]

# Display compact summaries (DataFrames limited to top 50 rows where applicable) and window counts
metric_summary, metric_window_count, trace_summary, trace_window_count, log_summary, log_window_count, error_summary, error_window_count

```
Out[1]:
```
```python
summary = (
    f"Summaries computed for all files (compact tables produced, top 50 rows shown where applicable).\n\n"
    f"1) metric.csv: per-(cmdb_id, kpi_name) count and global P95 computed. "
    f"Number of metric rows falling inside the incident window (2024-01-25 16:19:12 to 2024-01-25 16:49:12 UTC): {metric_window_count}.\n\n"
    f"2) trace.csv: per-(cmdb_id, trace_name) count and global P95 computed. "
    f"Number of trace rows inside the incident window: {trace_window_count}.\n\n"
    f"3) log.csv: per-(cmdb_id, log_name) count and global P95 computed. "
    f"Number of log rows inside the incident window: {log_window_count}.\n\n"
    f"4) error_logs.csv: no error_log rows were present in the summary (error_logs summary is empty), "
    f"and number of error_log rows inside the incident window: {error_window_count}.\n\n"
    f"Notes: The outputs are compact summary tables (counts and global P95 values) for metric/trace/log files; "
    f"error_logs contains no rows in the dataset. If you want specific top KPIs or the full summary tables, "
    f"ask to display metric_summary.head(n), trace_summary.head(n), log_summary.head(n), or error_summary.head(n)."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                        cmdb_id    kpi_name  count     p95_value
0   ts-admin-basic-info-service         cpu     25  7.476217e-01
1   ts-admin-basic-info-service      diskio     25  1.351411e+06
2   ts-admin-basic-info-service       error     25  0.000000e+00
3   ts-admin-basic-info-service  latency-50     25  1.795683e-02
4   ts-admin-basic-info-service  latency-90     25  2.833110e-02
..                          ...         ...    ...           ...
45              ts-auth-service  latency-50     25  2.599869e-01
46              ts-auth-service  latency-90     25  8.372433e-01
47              ts-auth-service         mem     25  2.384086e+08
48              ts-auth-service      socket     25  1.748333e+01
49              ts-auth-service    workload     25  2.751627e+00

[50 rows x 4 columns], 8975,    cmdb_id                                         trace_name  count  p95_value
0     root  trace.to_ts-admin-basic-info-service.duration_...     25   0.016881
1     root  trace.to_ts-admin-basic-info-service.duration_p95     25   0.040447
2     root    trace.to_ts-admin-basic-info-service.error_rate     25   0.000000
3     root     trace.to_ts-admin-basic-info-service.row_count     25  21.800000
4     root     trace.to_ts-admin-travel-service.duration_mean     25   0.210354
..     ...                                                ...    ...        ...
45    root          trace.to_ts-preserve-service.duration_p95     24   1.892761
46    root            trace.to_ts-preserve-service.error_rate     24   0.000000
47    root             trace.to_ts-preserve-service.row_count     24   6.000000
48    root          trace.to_ts-station-service.duration_mean      1   0.001264
49    root           trace.to_ts-station-service.duration_p95      1   0.001264

[50 rows x 4 columns], 18464,                         cmdb_id         log_name  count  p95_value
0   ts-admin-basic-info-service  log.error_count     25        0.0
1   ts-admin-basic-info-service    log.row_count     25       32.8
2       ts-admin-travel-service  log.error_count     25        0.0
3       ts-admin-travel-service    log.row_count     25       30.0
4            ts-assurance-mongo  log.error_count     24        0.0
..                          ...              ...    ...        ...
45       ts-order-other-service    log.row_count     25      487.0
46             ts-order-service  log.error_count     25        0.0
47             ts-order-service    log.row_count     25      404.6
48             ts-payment-mongo  log.error_count     24        0.0
49             ts-payment-mongo    log.row_count     24        1.0

[50 rows x 4 columns], 2182, Empty DataFrame
Columns: [cmdb_id, count, earliest_timestamp, latest_timestamp]
Index: [], 0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

# reuse existing variables: metric, window_start, window_end
import pandas as pd

cmdbs = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# restrict to selected cmdb_ids
metric_subset = metric[metric['cmdb_id'].isin(cmdbs)].copy()

# 1) compute global P95 and P05 per (cmdb_id, kpi_name) using full series (no time filter)
thresholds = (
    metric_subset
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(p95=lambda s: float(s.quantile(0.95)), p05=lambda s: float(s.quantile(0.05)))
)

# 2) filter metric rows to the incident window
metric_window = metric_subset[(metric_subset['timestamp'] >= window_start) & (metric_subset['timestamp'] <= window_end)].copy()

# merge thresholds into window rows so each row carries its global p95/p05
metric_window = metric_window.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# 3) group and compute required aggregated anomaly stats for rows in the window
def per_group_agg(df):
    p95 = df['p95'].iloc[0]
    p05 = df['p05'].iloc[0]
    total_points = len(df)
    # high anomalies: value > global P95
    high_mask = df['value'] > p95
    count_high = int(high_mask.sum())
    earliest_high = df.loc[high_mask, 'timestamp'].min() if count_high > 0 else pd.NaT
    max_in_window = float(df['value'].max()) if total_points > 0 else float('nan')
    # low anomalies: value <= global P05
    low_mask = df['value'] <= p05
    count_low = int(low_mask.sum())
    earliest_low = df.loc[low_mask, 'timestamp'].min() if count_low > 0 else pd.NaT
    min_in_window = float(df['value'].min()) if total_points > 0 else float('nan')
    return pd.Series({
        'p95': p95,
        'p05': p05,
        'total_points_in_window': total_points,
        'count_high_anom': count_high,
        'earliest_high_ts': earliest_high,
        'max_value_in_window': max_in_window,
        'count_low_anom': count_low,
        'earliest_low_ts': earliest_low,
        'min_value_in_window': min_in_window
    })

grouped = metric_window.groupby(['cmdb_id', 'kpi_name'], sort=False).apply(per_group_agg).reset_index()

# 4) keep only rows where total_points_in_window > 0, sort by anomaly counts desc, limit to top 50
grouped['total_anoms'] = grouped['count_high_anom'] + grouped['count_low_anom']
metric_anomaly_summary = (
    grouped[grouped['total_points_in_window'] > 0]
    .sort_values(by=['total_anoms', 'total_points_in_window'], ascending=[False, False])
    .drop(columns=['total_anoms'])
    .head(50)
    # ensure compact column order as requested
    [['cmdb_id','kpi_name','p95','p05','total_points_in_window',
      'count_high_anom','earliest_high_ts','max_value_in_window',
      'count_low_anom','earliest_low_ts','min_value_in_window']]
)

metric_anomaly_summary

```
Out[2]:
```
Summary of findings (metrics for the five requested services, window 2024-01-25 16:19:12–16:49:12 UTC):

- Returned 35 (cmdb_id, kpi_name) rows with data inside the incident window. Most metric series had 25 points in-window.
- Many KPIs show both high and low anomalies relative to their global baselines (global P95 and P05). Typical anomaly counts per KPI are small (commonly 2 high and 2 low points).
- ts-auth-service is the most prominently affected: multiple KPIs (cpu, diskio, latency-50, latency-90, mem, socket, workload, etc.) each show both high and low anomalies during the window. Example: ts-auth-service cpu — p95=14.71, p05=12.32, 25 points in window, 2 points >p95 (earliest high at 2024-01-25T16:30:00Z, max=17.65321) and 2 points <=p05 (earliest low at 2024-01-25T16:33:00Z, min=12.02072).
- Other services (ts-train-service, ts-travel-service, ts-order-service, ts-route-service) also show similar patterns across latency, mem, socket, workload, and diskio KPIs (again typically 1–2 highs and 1–2 lows each). Example: ts-travel-service latency-90 — p95=0.32548, p05=0.12431, 25 points, 2 highs (earliest 16:35Z) and 2 lows (earliest 16:26Z).
- Time pattern: high anomalies appear at various times across the window (e.g., ~16:22–16:37 UTC); low anomalies also occur across the window (e.g., ~16:23–16:35 UTC). There is not a single isolated spike time—multiple KPIs across services show anomalies at different minute stamps within the window.

Recommended next steps:
- Prioritize investigation of ts-auth-service (multiple KPIs anomalous) and then ts-train/ts-travel for correlated issues.
- Examine traces, logs, and infrastructure (CPU/memory/disk I/O, sockets, network) around the earliest_high_ts values shown for affected KPIs to find causal traces or error patterns.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name           p95           p05  total_points_in_window  count_high_anom          earliest_high_ts  max_value_in_window  count_low_anom           earliest_low_ts  min_value_in_window
0     ts-auth-service         cpu  1.471345e+01  1.231804e+01                      25                2 2024-01-25 16:30:00+00:00         1.765321e+01               2 2024-01-25 16:33:00+00:00         1.202072e+01
1     ts-auth-service      diskio  5.886898e+04  3.901405e+04                      25                2 2024-01-25 16:28:00+00:00         6.378122e+04               2 2024-01-25 16:33:00+00:00         3.539857e+04
2     ts-auth-service  latency-50  2.599869e-01  2.127208e-01                      25                2 2024-01-25 16:30:00+00:00         2.772676e-01               2 2024-01-25 16:32:00+00:00         2.030024e-01
3     ts-auth-service  latency-90  8.372433e-01  5.117547e-01                      25                2 2024-01-25 16:22:00+00:00         9.668592e-01               2 2024-01-25 16:34:00+00:00         4.759645e-01
4     ts-auth-service         mem  2.384086e+08  2.362365e+08                      25                2 2024-01-25 16:35:00+00:00         2.422649e+08               2 2024-01-25 16:22:00+00:00         2.361255e+08
..                ...         ...           ...           ...                     ...              ...                       ...                  ...             ...                       ...                  ...
31  ts-travel-service  latency-90  3.254834e-01  1.243113e-01                      25                2 2024-01-25 16:35:00+00:00         3.644258e-01               2 2024-01-25 16:26:00+00:00         1.034648e-01
32  ts-travel-service         mem  2.719004e+08  2.710996e+08                      25                2 2024-01-25 16:42:00+00:00         2.719098e+08               2 2024-01-25 16:23:00+00:00         2.710806e+08
33  ts-travel-service      socket  1.871333e+01  1.473333e+01                      25                2 2024-01-25 16:37:00+00:00         1.895000e+01               2 2024-01-25 16:27:00+00:00         1.448333e+01
34  ts-travel-service    workload  1.004225e+01  5.585287e+00                      25                2 2024-01-25 16:25:00+00:00         1.027003e+01               2 2024-01-25 16:35:00+00:00         4.880017e+00
5     ts-auth-service      socket  1.748333e+01  1.535692e+01                      25                1 2024-01-25 16:22:00+00:00         1.800000e+01               2 2024-01-25 16:34:00+00:00         1.523333e+01

[35 rows x 11 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import re
import pandas as pd

# reuse existing variables: trace, window_start, window_end

# target substrings
targets = ['to_ts-auth-service','to_ts-order-service','to_ts-route-service','to_ts-train-service','to_ts-travel-service']
pattern = '|'.join(re.escape(t) for t in targets)

# select trace rows whose trace_name contains any target substring (for full-series threshold calc)
mask_full = trace['trace_name'].str.contains(pattern, na=False)
trace_selected_full = trace[mask_full].copy()

# compute global P95 and P05 per trace_name using full available series (no time filter)
thresholds = (
    trace_selected_full
    .groupby('trace_name', as_index=False)['value']
    .agg(p95=lambda s: float(s.quantile(0.95)), p05=lambda s: float(s.quantile(0.05)))
)

# extract metric_type (suffix after last dot) for clarity
thresholds['metric_type'] = thresholds['trace_name'].str.rsplit('.', n=1).str[-1]

# filter trace rows to the incident window and to the selected trace_names
trace_window = trace_selected_full[(trace_selected_full['timestamp'] >= window_start) & (trace_selected_full['timestamp'] <= window_end)].copy()

# merge thresholds into window rows so each row carries its global p95/p05
trace_window = trace_window.merge(thresholds, on='trace_name', how='left')

# group and compute aggregated anomaly stats for rows in the window
def per_group_agg(df):
    p95 = df['p95'].iloc[0]
    p05 = df['p05'].iloc[0]
    total_points = len(df)
    high_mask = df['value'] > p95
    count_high = int(high_mask.sum())
    earliest_high = df.loc[high_mask, 'timestamp'].min() if count_high > 0 else pd.NaT
    max_in_window = float(df['value'].max()) if total_points > 0 else float('nan')
    low_mask = df['value'] <= p05
    count_low = int(low_mask.sum())
    earliest_low = df.loc[low_mask, 'timestamp'].min() if count_low > 0 else pd.NaT
    min_in_window = float(df['value'].min()) if total_points > 0 else float('nan')
    metric_type = df['metric_type'].iloc[0]
    return pd.Series({
        'metric_type': metric_type,
        'p95': p95,
        'p05': p05,
        'total_points_in_window': total_points,
        'count_high_anom': count_high,
        'earliest_high_ts': earliest_high,
        'max_value_in_window': max_in_window,
        'count_low_anom': count_low,
        'earliest_low_ts': earliest_low,
        'min_value_in_window': min_in_window
    })

grouped = trace_window.groupby('trace_name', sort=False).apply(per_group_agg).reset_index()

# keep only rows where total_points_in_window > 0, sort by anomaly counts desc and limit to top 20
grouped['total_anoms'] = grouped['count_high_anom'] + grouped['count_low_anom']
trace_anomaly_summary = (
    grouped[grouped['total_points_in_window'] > 0]
    .sort_values(by=['total_anoms','total_points_in_window'], ascending=[False, False])
    .drop(columns=['total_anoms'])
    .head(20)
    [['trace_name','metric_type','p95','p05','total_points_in_window',
      'count_high_anom','earliest_high_ts','max_value_in_window',
      'count_low_anom','earliest_low_ts','min_value_in_window']]
)

trace_anomaly_summary

```
Out[3]:
```
Summary of trace analysis for calls to the five services during 2024-01-25 16:19:12–16:49:12 UTC:

- What was analyzed: trace rows whose trace_name contains to_ts-auth-service, to_ts-order-service, to_ts-route-service, to_ts-train-service, or to_ts-travel-service. For each trace_name we computed global P95/P05 from the full series, then counted anomalies inside the incident window (value > global P95 = high anomaly; value <= global P05 = low anomaly).

Key findings (compact):

- Many duration and row_count traces show both high and low anomalies in the window, typically a few points each.
  - trace.to_ts-order-service.duration_mean and .duration_p95 (171 points in-window): 9 high anomalies (earliest high 2024-01-25T16:35:00Z) and 9 low anomalies (earliest low 2024-01-25T16:22:00Z). max ~1.762s, min ~0.0042s.
  - trace.to_ts-travel-service.duration_mean and .duration_p95 (149 points): 8 highs (earliest high 16:35Z) and 8 lows (earliest low ~16:23Z). max ~2.226s for p95 metric.
  - trace.to_ts-route-service.duration_mean/.duration_p95 (100 points): 5 highs (earliest high 16:35Z) and 5 lows (earliest low ~16:23–16:24Z).
  - trace.to_ts-train-service.duration_mean/.duration_p95 (100 points): 5 highs (earliest highs ~16:38–16:42Z) and 5 lows (earliest low ~16:27Z).
  - trace.to_ts-auth-service.duration_mean/.duration_p95 (50 points): 3 highs (earliest high ~16:29Z) and 3 lows (earliest low ~16:33–16:34Z).

- Row_count traces show large absolute values and anomalous counts:
  - trace.to_ts-route-service.row_count: 100 points, 5 highs (earliest high 16:25Z), 6 lows (earliest low 16:22Z), max ~2108, min 19.
  - trace.to_ts-train-service.row_count: 100 points, 4 highs (earliest 16:25Z), 6 lows (earliest 16:22Z), max ~2644, min 19.
  - trace.to_ts-order-service.row_count and trace.to_ts-travel-service.row_count also show multiple highs and lows (order: 8 highs, 9 lows; travel: 8 highs, 15 lows).

- Error_rate traces for all these targets show global P95 = P05 = 0 and every in-window value = 0:
  - trace.to_ts-*-service.error_rate rows have count_low_anom = total_points_in_window (e.g., order: 171 low points). This means error_rate is zero throughout the window (counted as “low” relative to a zero baseline) — not an error spike.

Interpretation and actionable points:

- There are clear duration spikes around ~16:35 UTC for order/travel/route services (seen in duration_mean and duration_p95 metrics). These coincide with increases in row_count for several services (peaks around ~16:25–16:35), suggesting a traffic or load change that affected latency.
- auth and train also show anomalies but with fewer points; auth has smaller series (50 points) and shows highs around ~16:29.
- error_rate is zero across the window, so request failures do not appear to be the issue—this looks like performance/latency and throughput fluctuation rather than increased error rates.

Recommended next steps:
- Investigate request/traffic surge or downstream dependency behavior around 16:25–16:35 UTC (especially for order, travel, route).
- Correlate these trace findings with metric anomalies (CPU, mem, socket, diskio) and logs at the earliest_high_ts values to find root cause (resource saturation, slow downstream calls, or throttling).

The original code execution output of IPython Kernel is also provided below for reference:

trace_name    metric_type          p95         p05  total_points_in_window  count_high_anom          earliest_high_ts  max_value_in_window  count_low_anom           earliest_low_ts  min_value_in_window
6       trace.to_ts-order-service.error_rate     error_rate     0.000000    0.000000                     171                0                       NaT             0.000000             171 2024-01-25 16:22:00+00:00             0.000000
10     trace.to_ts-travel-service.error_rate     error_rate     0.000000    0.000000                     149                0                       NaT             0.000000             149 2024-01-25 16:22:00+00:00             0.000000
14      trace.to_ts-route-service.error_rate     error_rate     0.000000    0.000000                     100                0                       NaT             0.000000             100 2024-01-25 16:22:00+00:00             0.000000
18      trace.to_ts-train-service.error_rate     error_rate     0.000000    0.000000                     100                0                       NaT             0.000000             100 2024-01-25 16:22:00+00:00             0.000000
2        trace.to_ts-auth-service.error_rate     error_rate     0.000000    0.000000                      50                0                       NaT             0.000000              50 2024-01-25 16:22:00+00:00             0.000000
11      trace.to_ts-travel-service.row_count      row_count  1390.000000    3.000000                     149                8 2024-01-25 16:24:00+00:00          1796.000000              15 2024-01-25 16:25:00+00:00             2.000000
4    trace.to_ts-order-service.duration_mean  duration_mean     0.549546    0.004973                     171                9 2024-01-25 16:35:00+00:00             0.827044               9 2024-01-25 16:22:00+00:00             0.004196
5     trace.to_ts-order-service.duration_p95   duration_p95     1.103439    0.006045                     171                9 2024-01-25 16:35:00+00:00             1.762091               9 2024-01-25 16:22:00+00:00             0.004196
7        trace.to_ts-order-service.row_count      row_count   411.000000    2.500000                     171                8 2024-01-25 16:25:00+00:00           474.000000               9 2024-01-25 16:22:00+00:00             1.000000
8   trace.to_ts-travel-service.duration_mean  duration_mean     1.058362    0.009694                     149                8 2024-01-25 16:35:00+00:00             1.451085               8 2024-01-25 16:23:00+00:00             0.009367
9    trace.to_ts-travel-service.duration_p95   duration_p95     1.897599    0.011487                     149                8 2024-01-25 16:35:00+00:00             2.225844               8 2024-01-25 16:23:00+00:00             0.010612
15       trace.to_ts-route-service.row_count      row_count  1930.300000  109.000000                     100                5 2024-01-25 16:25:00+00:00          2108.000000               6 2024-01-25 16:22:00+00:00            19.000000
12   trace.to_ts-route-service.duration_mean  duration_mean     0.004674    0.001283                     100                5 2024-01-25 16:35:00+00:00             0.005953               5 2024-01-25 16:23:00+00:00             0.001270
13    trace.to_ts-route-service.duration_p95   duration_p95     0.004719    0.001737                     100                5 2024-01-25 16:35:00+00:00             0.009703               5 2024-01-25 16:24:00+00:00             0.001697
16   trace.to_ts-train-service.duration_mean  duration_mean     0.006784    0.001064                     100                5 2024-01-25 16:38:00+00:00             0.018818               5 2024-01-25 16:27:00+00:00             0.001040
17    trace.to_ts-train-service.duration_p95   duration_p95     0.006547    0.001508                     100                5 2024-01-25 16:42:00+00:00             0.105178               5 2024-01-25 16:27:00+00:00             0.001465
19       trace.to_ts-train-service.row_count      row_count  2384.000000  109.000000                     100                4 2024-01-25 16:25:00+00:00          2644.000000               6 2024-01-25 16:22:00+00:00            19.000000
0     trace.to_ts-auth-service.duration_mean  duration_mean     0.365718    0.001844                      50                3 2024-01-25 16:29:00+00:00             0.394993               3 2024-01-25 16:34:00+00:00             0.001484
1      trace.to_ts-auth-service.duration_p95   duration_p95     0.955933    0.001951                      50                3 2024-01-25 16:29:00+00:00             1.013001               3 2024-01-25 16:33:00+00:00             0.001837
3         trace.to_ts-auth-service.row_count      row_count   329.100000   69.800000                      50                3 2024-01-25 16:33:00+00:00           342.000000               3 2024-01-25 16:22:00+00:00            14.000000```
```

In [ ]:
# In[4]:

# reuse existing variables: log, window_start, window_end
import pandas as pd

cmdbs = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# restrict to selected cmdb_ids (full series for threshold calculation)
log_subset = log[log['cmdb_id'].isin(cmdbs)].copy()

# 1) compute global P95 and P05 per (cmdb_id, log_name) using full series
log_thresholds = (
    log_subset
    .groupby(['cmdb_id', 'log_name'], as_index=False)['value']
    .agg(p95=lambda s: float(s.quantile(0.95)), p05=lambda s: float(s.quantile(0.05)))
)

# 2) filter log rows to the incident window
log_window = log_subset[(log_subset['timestamp'] >= window_start) & (log_subset['timestamp'] <= window_end)].copy()

# merge thresholds into window rows
log_window = log_window.merge(log_thresholds, on=['cmdb_id', 'log_name'], how='left')

# 3) aggregate per (cmdb_id, log_name)
def agg_logs(df):
    p95 = df['p95'].iloc[0]
    p05 = df['p05'].iloc[0]
    total_points = len(df)
    high_mask = df['value'] > p95
    count_high = int(high_mask.sum())
    earliest_high = df.loc[high_mask, 'timestamp'].min() if count_high > 0 else pd.NaT
    max_in_window = float(df['value'].max()) if total_points > 0 else float('nan')
    low_mask = df['value'] <= p05
    count_low = int(low_mask.sum())
    earliest_low = df.loc[low_mask, 'timestamp'].min() if count_low > 0 else pd.NaT
    min_in_window = float(df['value'].min()) if total_points > 0 else float('nan')
    return pd.Series({
        'p95': p95,
        'p05': p05,
        'total_points_in_window': total_points,
        'count_high_anom': count_high,
        'earliest_high_ts': earliest_high,
        'max_value_in_window': max_in_window,
        'count_low_anom': count_low,
        'earliest_low_ts': earliest_low,
        'min_value_in_window': min_in_window
    })

log_grouped = log_window.groupby(['cmdb_id','log_name'], sort=False).apply(agg_logs).reset_index()

# 4) keep only rows with data in window, sort by anomaly counts desc, limit top 20, and present requested columns
log_grouped['total_anoms'] = log_grouped['count_high_anom'] + log_grouped['count_low_anom']

log_anomaly_summary = (
    log_grouped[log_grouped['total_points_in_window'] > 0]
    .sort_values(by=['total_anoms', 'total_points_in_window'], ascending=[False, False])
    .drop(columns=['total_anoms'])
    .head(20)
    [['cmdb_id','log_name','p95','p05','total_points_in_window',
      'count_high_anom','earliest_high_ts','max_value_in_window',
      'count_low_anom','earliest_low_ts','min_value_in_window']]
)

log_anomaly_summary

```
Out[4]:
```
Summary of log analysis for the five services during 2024-01-25 16:19:12–16:49:12 UTC:

Common note: each service has 25 log points inside the window.

- Overall: log.error_count is zero for all services (global P95=P05=0). There are no high error spikes; all 25 points per service are at the zero baseline.

Per-service log.row_count (global P95 / P05, window stats):
- ts-auth-service — row_count: p95=331.6, p05=275.2. In-window: 25 points; 2 high anomalies (earliest high 2024-01-25T16:33:00Z, max=342.0), 2 low anomalies (earliest low 2024-01-25T16:22:00Z, min=56.0).
- ts-order-service — row_count: p95=404.6, p05=230.6. In-window: 25 points; 2 highs (earliest high 2024-01-25T16:31:00Z, max=423.0), 2 lows (earliest low 2024-01-25T16:35:00Z, min=42.0).
- ts-route-service — row_count: p95=1036.4, p05=840.2. In-window: 25 points; 2 highs (earliest high 2024-01-25T16:25:00Z, max=1054.0), 2 lows (earliest low 2024-01-25T16:22:00Z, min=152.0).
- ts-travel-service — row_count: p95=1869.6, p05=1019.0. In-window: 25 points; 2 highs (earliest high 2024-01-25T16:25:00Z, max=2026.0), 2 lows (earliest low 2024-01-25T16:35:00Z, min=236.0).

Interpretation / takeaway:
- There are clear fluctuations in request/row counts (spikes and dips) around ~16:22–16:35 UTC across these services. These throughput changes align with trace and metric anomalies you previously observed.
- Error rates are zero, so the issue looks like performance/traffic variability (load/throughput) rather than increased request failures.

Recommended next steps:
- Correlate the row_count highs (around ~16:25 and ~16:31–16:33–16:35 UTC) with the trace latency spikes and metric anomalies (CPU/memory/socket) at the same times to find the root cause (traffic surge, downstream slowdowns, or resource saturation).

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id         log_name     p95     p05  total_points_in_window  count_high_anom          earliest_high_ts  max_value_in_window  count_low_anom           earliest_low_ts  min_value_in_window
0    ts-auth-service  log.error_count     0.0     0.0                      25                0                       NaT                  0.0              25 2024-01-25 16:22:00+00:00                  0.0
2   ts-order-service  log.error_count     0.0     0.0                      25                0                       NaT                  0.0              25 2024-01-25 16:22:00+00:00                  0.0
4   ts-route-service  log.error_count     0.0     0.0                      25                0                       NaT                  0.0              25 2024-01-25 16:22:00+00:00                  0.0
6  ts-travel-service  log.error_count     0.0     0.0                      25                0                       NaT                  0.0              25 2024-01-25 16:22:00+00:00                  0.0
1    ts-auth-service    log.row_count   331.6   275.2                      25                2 2024-01-25 16:33:00+00:00                342.0               2 2024-01-25 16:22:00+00:00                 56.0
3   ts-order-service    log.row_count   404.6   230.6                      25                2 2024-01-25 16:31:00+00:00                423.0               2 2024-01-25 16:35:00+00:00                 42.0
5   ts-route-service    log.row_count  1036.4   840.2                      25                2 2024-01-25 16:25:00+00:00               1054.0               2 2024-01-25 16:22:00+00:00                152.0
7  ts-travel-service    log.row_count  1869.6  1019.0                      25                2 2024-01-25 16:25:00+00:00               2026.0               2 2024-01-25 16:35:00+00:00                236.0```
```

In [ ]:
# In[5]:

# reuse existing variables: metric, window_start, window_end
import numpy as np
import pandas as pd

# filter to ts-order-service
metric_order = metric[metric['cmdb_id'] == "ts-order-service"].copy()

# 1) global P95 and P05 per kpi_name using full series (no time filter)
thresholds = (
    metric_order
    .groupby('kpi_name', as_index=False)['value']
    .agg(p95=lambda s: float(s.quantile(0.95)), p05=lambda s: float(s.quantile(0.05)))
)

# 2) filter rows to incident window
metric_order_window = metric_order[(metric_order['timestamp'] >= window_start) & (metric_order['timestamp'] <= window_end)].copy()

# merge thresholds into window rows
metric_order_window = metric_order_window.merge(thresholds, on='kpi_name', how='left')

# 3) compute aggregated stats per kpi_name
def agg_kpi(df):
    p95 = df['p95'].iloc[0]
    p05 = df['p05'].iloc[0]
    total_points = len(df)
    # highs
    high_mask = df['value'] > p95
    count_high = int(high_mask.sum())
    earliest_high = df.loc[high_mask, 'timestamp'].min() if count_high > 0 else pd.NaT
    max_in_window = float(df['value'].max()) if total_points > 0 else float('nan')
    if count_high > 0 and p95 != 0:
        breach_ratio_high = float((max_in_window - p95) / p95)
    else:
        breach_ratio_high = 0.0
    # lows
    low_mask = df['value'] <= p05
    count_low = int(low_mask.sum())
    earliest_low = df.loc[low_mask, 'timestamp'].min() if count_low > 0 else pd.NaT
    min_in_window = float(df['value'].min()) if total_points > 0 else float('nan')
    if count_low > 0 and p05 != 0:
        breach_ratio_low = float((p05 - min_in_window) / p05)
    else:
        breach_ratio_low = 0.0
    return pd.Series({
        'p95': p95,
        'p05': p05,
        'total_points_in_window': total_points,
        'count_high_anom': count_high,
        'earliest_high_ts': earliest_high,
        'max_value_in_window': max_in_window,
        'breach_ratio_high': breach_ratio_high,
        'count_low_anom': count_low,
        'earliest_low_ts': earliest_low,
        'min_value_in_window': min_in_window,
        'breach_ratio_low': breach_ratio_low
    })

order_kpi_anomalies = (
    metric_order_window
    .groupby('kpi_name', sort=False)
    .apply(agg_kpi)
    .reset_index()
)

# keep only KPIs that had any points in window
order_kpi_anomalies = order_kpi_anomalies[order_kpi_anomalies['total_points_in_window'] > 0]

# 4) sort by breach_ratio_high desc and limit to top 20, ensure requested column order
order_kpi_anomalies = (
    order_kpi_anomalies
    .sort_values(by='breach_ratio_high', ascending=False)
    .head(20)
    [['kpi_name','p95','p05','total_points_in_window',
      'count_high_anom','earliest_high_ts','max_value_in_window','breach_ratio_high',
      'count_low_anom','earliest_low_ts','min_value_in_window','breach_ratio_low']]
)

order_kpi_anomalies

```
Out[5]:
```
Summary for ts-order-service (window 2024-01-25 16:19:12–16:49:12 UTC):

- General: Each KPI had 25 points in the window. For every KPI shown there are 2 points above the global P95 and 2 points at/below the global P05 (small number of anomalies, both high and low).

Top KPI highlights (sorted by how far the worst high point exceeded the global P95):

1. latency-90
   - global P95 = 0.4866 s, P05 = 0.01578 s
   - in-window max = 0.5833 s (breach_ratio_high ≈ 0.199 → ~19.9% above P95)
   - earliest high at 2024-01-25T16:35:00Z
   - min in-window ≈ 0.01443 s (breach_ratio_low ≈ 0.085)

2. mem
   - global P95 ≈ 936,238,600, P05 ≈ 261,039,600
   - in-window max ≈ 1,002,420,000 (breach_ratio_high ≈ 0.0707)
   - earliest high at 2024-01-25T16:39:00Z
   - min in-window ≈ 260,530,700 (small low breach)

3. workload
   - global P95 ≈ 6.6501, P05 ≈ 3.7150
   - in-window max ≈ 7.0212 (breach_ratio_high ≈ 0.0558)
   - earliest high at 2024-01-25T16:25:00Z
   - min in-window ≈ 2.9811 (breach_ratio_low ≈ 0.1976 → ~19.8% below P05), earliest low 16:35Z

4. socket
   - P95 ≈ 17.7267, P05 ≈ 13.39
   - max ≈ 18.25 (breach_ratio_high ≈ 0.0295), earliest high 16:36Z
   - min ≈ 12.55 (breach_ratio_low ≈ 0.0627), earliest low 16:27Z

5. latency-50
   - P95 ≈ 0.04813 s, P05 ≈ 0.007118 s
   - max ≈ 0.04939 s (breach_ratio_high ≈ 0.0260), earliest high 16:36Z
   - min ≈ 0.007061 s (small low breach), earliest low 16:31Z

6. cpu
   - P95 ≈ 39.4547, P05 ≈ 1.5844
   - max ≈ 39.5855 (tiny breach_ratio_high ≈ 0.0033), earliest high 16:39Z
   - min ≈ 1.3042 (breach_ratio_low ≈ 0.1769), earliest low 16:30Z

7. diskio
   - P95 ≈ 27,037,570, P05 ≈ 50,124
   - max ≈ 27,118,990 (breach_ratio_high ≈ 0.0030), earliest high 16:36Z
   - min ≈ 46,705 (breach_ratio_low ≈ 0.0682), earliest low 16:26Z

Interpretation and suggested next steps:
- The most significant relative high breach is latency-90 (~20% above its P95) with earliest high at 16:35 UTC — this aligns with trace and row_count spikes seen earlier. Memory also shows a notable absolute increase (earliest high 16:39 UTC).
- Many KPIs show only 2 high and 2 low points in-window, so anomalies are brief but consistent across metrics.
- Investigate the 16:25–16:40 UTC period (especially ~16:35–16:39 UTC): correlate these KPI high timestamps with trace latency spikes, row_count peaks, and logs to determine if a traffic surge or resource contention caused elevated latencies and memory usage.

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name           p95           p05  total_points_in_window  count_high_anom          earliest_high_ts  max_value_in_window  breach_ratio_high  count_low_anom           earliest_low_ts  min_value_in_window  breach_ratio_low
3  latency-90  4.866187e-01  1.577707e-02                      25                2 2024-01-25 16:35:00+00:00         5.832655e-01           0.198609               2 2024-01-25 16:22:00+00:00         1.443211e-02          0.085247
4         mem  9.362386e+08  2.610396e+08                      25                2 2024-01-25 16:39:00+00:00         1.002420e+09           0.070689               2 2024-01-25 16:22:00+00:00         2.605307e+08          0.001949
6    workload  6.650120e+00  3.715016e+00                      25                2 2024-01-25 16:25:00+00:00         7.021167e+00           0.055795               2 2024-01-25 16:35:00+00:00         2.981083e+00          0.197559
5      socket  1.772667e+01  1.339000e+01                      25                2 2024-01-25 16:36:00+00:00         1.825000e+01           0.029522               2 2024-01-25 16:27:00+00:00         1.255000e+01          0.062733
2  latency-50  4.813434e-02  7.118434e-03                      25                2 2024-01-25 16:36:00+00:00         4.938779e-02           0.026041               2 2024-01-25 16:31:00+00:00         7.061115e-03          0.008052
0         cpu  3.945472e+01  1.584438e+00                      25                2 2024-01-25 16:39:00+00:00         3.958549e+01           0.003314               2 2024-01-25 16:30:00+00:00         1.304177e+00          0.176883
1      diskio  2.703757e+07  5.012472e+04                      25                2 2024-01-25 16:36:00+00:00         2.711899e+07           0.003011               2 2024-01-25 16:26:00+00:00         4.670575e+04          0.068209```
```

In [ ]:
# In[6]:

# reuse existing variables: metric, window_start, window_end
import numpy as np
import pandas as pd

cmdbs = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# 1) compute global P95 per (cmdb_id, kpi_name) using full series (no time filter)
thresholds = (
    metric[metric['cmdb_id'].isin(cmdbs)]
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .agg(p95=lambda s: float(s.quantile(0.95)))
)

# 2) filter metric rows to the incident window for the selected cmdb_ids
metric_window = metric[
    (metric['cmdb_id'].isin(cmdbs)) &
    (metric['timestamp'] >= window_start) &
    (metric['timestamp'] <= window_end)
].merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

# 3) mark highs and compute per-row breach (handle p95==0)
metric_window['is_high'] = metric_window['value'] > metric_window['p95']
metric_window['breach'] = np.where(
    metric_window['is_high'],
    np.where(metric_window['p95'] > 0,
             (metric_window['value'] - metric_window['p95']) / metric_window['p95'],
             np.inf),
    np.nan
)

# earliest_high_ts per cmdb_id (UTC) and number of distinct KPIs with any high anomaly
earliest_high = metric_window[metric_window['is_high']].groupby('cmdb_id', as_index=True)['timestamp'].min()
num_high_kpis = metric_window[metric_window['is_high']].groupby('cmdb_id', as_index=True)['kpi_name'].nunique()

# max_breach per cmdb_id (use 0 when no high anomalies)
max_breach = metric_window.groupby('cmdb_id', as_index=True)['breach'].max()

# assemble results for all target cmdbs, filling missing with defaults
result = pd.DataFrame({'cmdb_id': cmdbs}).set_index('cmdb_id')
result['earliest_high_ts'] = earliest_high
result['max_breach'] = max_breach
result['number_of_high_anomalous_kpis'] = num_high_kpis

# fill NaNs: earliest_high_ts keep NaT; max_breach and counts fill with 0
result['max_breach'] = result['max_breach'].fillna(0.0)
result['number_of_high_anomalous_kpis'] = result['number_of_high_anomalous_kpis'].fillna(0).astype(int)

# reset index for presentation and sort by earliest_high_ts ascending (NaT last)
cmdb_high_summary = result.reset_index().sort_values('earliest_high_ts', na_position='last').head(5)

cmdb_high_summary

```
Out[6]:
```
Summary (metrics during 2024-01-25 16:19:12–16:49:12 UTC):

- ts-auth-service — earliest high: 2024-01-25T16:22:00Z; max_breach ≈ 0.1998 (≈19.98% above KPI P95); 7 distinct KPIs had at least one value > their P95 in-window.
- ts-route-service — earliest high: 2024-01-25T16:22:00Z; max_breach ≈ 0.0586 (≈5.86%); 7 KPIs.
- ts-train-service — earliest high: 2024-01-25T16:22:00Z; max_breach ≈ 0.0687 (≈6.87%); 7 KPIs.
- ts-travel-service — earliest high: 2024-01-25T16:22:00Z; max_breach ≈ 0.1196 (≈11.96%); 7 KPIs.
- ts-order-service — earliest high: 2024-01-25T16:25:00Z; max_breach ≈ 0.1986 (≈19.86%); 7 KPIs.

Interpretation and suggested focus:
- All five services had multiple KPIs exceed their global P95 during the incident window. The earliest detected highs mostly start at ~16:22 UTC (order starts ~16:25).
- The largest relative breaches were in ts-auth-service and ts-order-service (~20% above P95), so prioritize investigating those services first (correlate with traces, row_count, and logs around their earliest_high_ts).

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id          earliest_high_ts  max_breach  number_of_high_anomalous_kpis
0    ts-auth-service 2024-01-25 16:22:00+00:00    0.199801                              7
2   ts-route-service 2024-01-25 16:22:00+00:00    0.058623                              7
3   ts-train-service 2024-01-25 16:22:00+00:00    0.068694                              7
4  ts-travel-service 2024-01-25 16:22:00+00:00    0.119645                              7
1   ts-order-service 2024-01-25 16:25:00+00:00    0.198609                              7```
```